# 11. Does the efficient shape also win identifiability, not just cost?

Notebook 06 found that the "shipped" default draw (`nudge_shape="uniform"`)
wastes most of the band it's given -- the mean absolute value of a uniform
draw is only half its cap, so a `+/-80%` permission moves a typical week by
far less than 80%. A smarter shape, `edge` (every week moves by exactly the
cap) with `balance_signs=True` (equal up/down weeks per month, so the
monthly rescale barely has to move anything), gets 2-3x the variance-
reduction at the same nominal setting -- enough that all-Blackout fell off
notebook 06's own cost/gain frontier for variance alone.

**But every lever tested in notebooks 07 through 10 used `uniform`.** Bias
(07), saturation (09), adstock (08), and the benefit table that assembled
them (10) all compared Blackout against the wasteful default, never against
`edge`+`balance_signs=True` -- both real, already-shipped parameters on
`_generate_phased_schedule`, not something invented for this notebook. So it
was never known whether Blackout's clean sweep of every rigor column in
notebook 10 reflects something genuinely special about true zero-spend
weeks, or reflects being compared to a strawman.

**Headline result, stated up front because it's decisive:** `edge`+balanced
at the SAME nominal `+/-80%` cap does not just close the gap to Blackout --
it beats Blackout on bias, variance, saturation AND adstock, at slightly
LOWER cost. Blackout turns out to be dominated on every axis measured, not
merely expensive. Sections 2-5 below are the four checks, run with the exact
same methods as notebooks 06/07/08/09, with `edge`+balanced added at
`+/-80%` (matched nominal cap, the direct test) and `+/-40%` (notebook 06's
own "roughly matches uniform `+/-80%`" point, at half the setting).

In [ ]:
FAST_MODE = False

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from how_wrong_is_your_mmm import (
    apply_adstock,
    calibrate_baseline,
    fit_ols,
    simulate_demand,
    simulate_sales,
    simulate_spend,
)
from how_wrong_is_your_mmm._phaser import Blackout, _generate_phased_schedule

N_HIST, N_PLAN = 208, 52
CHANNELS = ["tv", "meta", "search"]
TRUE_MR = {"tv": 0.5, "meta": 1.0, "search": 1.5}
BASELINE_SHARE = 0.72
BASELINE_CV = 0.05

TRUTHS = [(0.6, 0.5), (0.8, 0.3)]
DECAYS = (0.0, 0.3, 0.5, 0.7)

if FAST_MODE:
    BV_N_SIMS, BV_N_PHASING_SEEDS, BV_N_DEMAND_SEEDS = 5, 1, 1
    SAT_N_SIMS, SAT_N_PHASING_SEEDS, SAT_N_DEMAND_SEEDS = 3, 1, 2
    AD_N_SIMS, AD_N_PHASING_SEEDS, AD_N_DEMAND_SEEDS = 5, 1, 2
    N_DRAW_SEEDS = 5
    # Coarser than the real grid below -- structural smoke-test only, never
    # cited, same FAST_MODE convention as notebook 09.
    B_CANDIDATES = np.round(np.linspace(0.20, 1.00, 7), 4)
    LAM_CANDIDATES = np.round(np.linspace(0.00, 0.90, 7), 4)
else:
    BV_N_SIMS, BV_N_PHASING_SEEDS, BV_N_DEMAND_SEEDS = 40, 6, 6
    SAT_N_SIMS, SAT_N_PHASING_SEEDS, SAT_N_DEMAND_SEEDS = 25, 6, 16
    AD_N_SIMS, AD_N_PHASING_SEEDS, AD_N_DEMAND_SEEDS = 40, 6, 16
    N_DRAW_SEEDS = 200
    B_CANDIDATES = np.round(np.linspace(0.20, 1.00, 33), 4)
    LAM_CANDIDATES = np.round(np.linspace(0.00, 0.90, 31), 4)

## 1. The shapes

`_generate_phased_schedule`'s `nudge_shape` controls how a week's deviation
magnitude is drawn within a symmetric `+/-cap` range: `"uniform"` (the
default -- magnitude ~ U(0, cap), so the mean realised move is half the cap)
or `"edge"` (magnitude = cap exactly, every week uses the whole permission).
`balance_signs=True` gives each month equal numbers of up- and down-weeks
instead of an independent coin flip, which keeps the monthly rescale close
to 1 and so keeps realised deviations close to what was actually drawn (see
notebook 06 for the full derivation and the overshoot problem this fixes).

Every world below uses the same fixed, session-44 windowing-corrected
`build_world` as notebooks 08-10 -- `white_noise` demand, `demand_share=1.0`,
`correlation=0.7`, the same corner used throughout this project.

In [ ]:
def _window_standardised(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std()


def build_world(demand_seed, process="white_noise", correlation=0.7, demand_share=1.0):
    n = N_HIST + N_PLAN
    demand = simulate_demand(n, process=process, seed=demand_seed)
    hist_demand = _window_standardised(demand[:N_HIST])
    plan_demand = _window_standardised(demand[N_HIST:])
    history = simulate_spend(
        n_obs=N_HIST, correlation=correlation, seed=1000 + demand_seed,
        start_date="2019-01-07", demand=hist_demand, demand_share=demand_share,
    )
    plan = simulate_spend(
        n_obs=N_PLAN, correlation=correlation, seed=2000 + demand_seed,
        start_date="2023-01-09", demand=plan_demand, demand_share=demand_share,
    )
    index = history.index.append(plan.index)
    demand_series = pd.Series(
        np.concatenate([hist_demand, plan_demand]), index=index, name="demand"
    )
    calibration = calibrate_baseline(
        pd.concat([history, plan]), TRUE_MR,
        baseline_share=BASELINE_SHARE, baseline_cv=BASELINE_CV,
    )
    return plan, demand_series, calibration


def all_channels(nominal):
    return {ch: nominal for ch in CHANNELS}


def is_unphased(spec):
    return all(isinstance(v, float) and v == 0.0 for v in spec.values())


def schedule_for(plan_df, spec, seed, nudge_shape="uniform", balance_signs=False, freq="M"):
    if is_unphased(spec):
        return plan_df
    return _generate_phased_schedule(
        plan_df, plan_df.index.to_period(freq).to_numpy(), alpha=1.0,
        max_weekly_deviation_pct=spec, seed=seed,
        nudge_shape=nudge_shape, balance_signs=balance_signs,
    )


# (label, spec, nudge_shape, balance_signs) -- one unified lever list, used
# by every section below.
LEVERS = [
    ("unphased", all_channels(0.0), "uniform", False),
    ("+/-80% (uniform)", all_channels(80.0), "uniform", False),
    ("+/-40% (edge, balanced)", all_channels(40.0), "edge", True),
    ("+/-80% (edge, balanced)", all_channels(80.0), "edge", True),
    ("Blackout", {ch: Blackout(max_dark_weeks_per_month=1) for ch in CHANNELS}, "uniform", False),
]

## 2. Bias and variance

Same method as notebook 07 / `grid_sweep.py`'s `sweep()`: fit OLS with
demand omitted, `N_DEMAND_SEEDS` worlds x `N_PHASING_SEEDS` schedule draws
each, `N_SIMS` sales-noise draws per schedule. One corner (`white_noise`,
`demand_share=1.0`, `correlation=0.7`) -- the same one notebook 07 cell 17's
committed numbers come from, so `+/-80% (uniform)` and `Blackout` here are a
direct, independently-recomputed check against that table, not a citation.

In [ ]:
def fit_draws(spend, demand_series, calibration, n_sims):
    aligned = demand_series.loc[spend.index]
    demand_values = aligned.to_numpy()
    draws = {ch: [] for ch in CHANNELS}
    for sim in range(n_sims):
        sales = simulate_sales(
            spend, TRUE_MR, base_sales=calibration.baseline_level, seed=sim,
            demand=demand_values, demand_coef=calibration.demand_coef,
        )
        fitted = fit_ols(spend, sales, controls=None)
        for ch in CHANNELS:
            draws[ch].append(fitted[ch])
    return {ch: np.array(v) for ch, v in draws.items()}


bv_rows = []
for label, spec, nudge_shape, balance_signs in LEVERS:
    bias = {ch: [] for ch in CHANNELS}
    cv = {ch: [] for ch in CHANNELS}
    for demand_seed in range(BV_N_DEMAND_SEEDS):
        plan_df, dem, cal = build_world(demand_seed)
        means = {ch: [] for ch in CHANNELS}
        spreads = {ch: [] for ch in CHANNELS}
        seeds = [0] if is_unphased(spec) else range(BV_N_PHASING_SEEDS)
        for seed in seeds:
            schedule = schedule_for(plan_df, spec, seed, nudge_shape, balance_signs)
            draws = fit_draws(schedule, dem, cal, BV_N_SIMS)
            for ch in CHANNELS:
                means[ch].append(draws[ch].mean())
                spreads[ch].append(100 * draws[ch].std() / abs(draws[ch].mean()))
        for ch in CHANNELS:
            bias[ch].append(100 * (np.mean(means[ch]) - TRUE_MR[ch]) / TRUE_MR[ch])
            cv[ch].append(np.median(spreads[ch]))
    bv_rows.append({
        "lever": label,
        "mean_bias_%": float(np.mean([np.mean(bias[ch]) for ch in CHANNELS])),
        "mean_cv_%": float(np.mean([np.mean(cv[ch]) for ch in CHANNELS])),
    })
    print(f"  done {label}", flush=True)

bias_variance = pd.DataFrame(bv_rows).set_index("lever")
ref_bias = bias_variance.loc["unphased", "mean_bias_%"]
ref_cv = bias_variance.loc["unphased", "mean_cv_%"]
bias_variance["bias_removed_%"] = 100 * (ref_bias - bias_variance["mean_bias_%"]) / ref_bias
bias_variance["cv_narrowed_%"] = 100 * (ref_cv - bias_variance["mean_cv_%"]) / ref_cv
pd.set_option("display.width", 200)
print(bias_variance.round(2).to_string())

**Real numbers (`FAST_MODE=False`):** `+/-80% (edge, balanced)` removes
81.2% of bias and narrows CV by 72.3% -- against `+/-80% (uniform)`'s 58.2%
/ 57.2% and Blackout's 74.1% / 62.9%. **Edge+balanced beats Blackout on both
axes, at the same nominal cap.** Even `+/-40% (edge, balanced)` -- half the
setting -- narrows CV by 56.9%, matching `+/-80% (uniform)`'s 57.2%, though
its bias removal (48.7%) doesn't fully keep pace at that lower setting.

## 3. Saturation identifiability

Same profile-likelihood method as notebook 09 section 3: grid over
`(b, lambda)`, fit at every candidate, keep the RSS-minimising point and the
spread across draws. Demand is a control throughout, so this stays about
whether the spend PATTERN identifies curvature, not about omitted-variable
bias. Both truths, same as notebook 09.

In [ ]:
def design(spend_arr, demand_arr, b, lam):
    n, k = spend_arr.shape
    X = np.empty((n, k + 2))
    X[:, 0] = 1.0
    for j in range(k):
        X[:, 1 + j] = apply_adstock(spend_arr[:, j], lam) ** b
    X[:, -1] = demand_arr
    return X


def profile(spend_arr, demand_arr, sales_matrix):
    n_sims = sales_matrix.shape[1]
    best_rss = np.full(n_sims, np.inf)
    best_b = np.zeros(n_sims)
    best_lam = np.zeros(n_sims)
    surface = np.zeros((len(B_CANDIDATES), len(LAM_CANDIDATES)))
    for i, b in enumerate(B_CANDIDATES):
        for j, lam in enumerate(LAM_CANDIDATES):
            X = design(spend_arr, demand_arr, b, lam)
            beta, _res, _rank, _sv = np.linalg.lstsq(X, sales_matrix, rcond=None)
            resid = sales_matrix - X @ beta
            rss = (resid**2).sum(axis=0)
            surface[i, j] = rss.mean()
            better = rss < best_rss
            best_rss = np.where(better, rss, best_rss)
            best_b = np.where(better, b, best_b)
            best_lam = np.where(better, lam, best_lam)
    return best_b, best_lam, surface


def valley_width(surface, tol=0.01):
    lo = surface.min()
    return float((surface <= lo * (1.0 + tol)).mean())


sat_rows = []
for b_true, lam_true in TRUTHS:
    for label, spec, nudge_shape, balance_signs in LEVERS:
        rec_b, rec_lam, widths = [], [], []
        for demand_seed in range(SAT_N_DEMAND_SEEDS):
            plan_df, dem, cal = build_world(demand_seed)
            ref = {ch: float(plan_df[ch].mean()) for ch in CHANNELS}
            seeds = [0] if is_unphased(spec) else range(SAT_N_PHASING_SEEDS)
            for phasing_seed in seeds:
                sched = schedule_for(plan_df, spec, phasing_seed, nudge_shape, balance_signs)
                dem_arr = dem.loc[sched.index].to_numpy()
                sales_cols = [
                    simulate_sales(
                        sched, TRUE_MR, base_sales=cal.baseline_level, seed=sim,
                        demand=dem_arr, demand_coef=cal.demand_coef,
                        saturation=b_true, adstock=lam_true, reference_spend=ref,
                    ).to_numpy()
                    for sim in range(SAT_N_SIMS)
                ]
                sales_matrix = np.column_stack(sales_cols)
                spend_arr = sched[CHANNELS].to_numpy()
                bb, ll, surface = profile(spend_arr, dem_arr, sales_matrix)
                rec_b.append(bb)
                rec_lam.append(ll)
                widths.append(valley_width(surface))
        rec_b = np.concatenate(rec_b)
        rec_lam = np.concatenate(rec_lam)
        sat_rows.append({
            "b_true": b_true, "lam_true": lam_true, "lever": label,
            "b_sd": rec_b.std(), "lam_sd": rec_lam.std(),
            "valley_%": 100 * float(np.mean(widths)),
        })
        print(f"  done b={b_true} lam={lam_true} {label}", flush=True)

saturation = pd.DataFrame(sat_rows)
for (b_true, lam_true), block in saturation.groupby(["b_true", "lam_true"]):
    print(f"\n=== true b = {b_true}, true lambda = {lam_true} ===")
    print(block[["lever", "b_sd", "lam_sd", "valley_%"]].round(3).to_string(index=False))

**Real numbers (`FAST_MODE=False`):**

| lever | (0.6,0.5) b_sd | valley_% | (0.8,0.3) b_sd | valley_% |
|---|---|---|---|---|
| +/-80% (uniform) | 0.330 | 8.38 | 0.276 | 5.01 |
| +/-40% (edge, balanced) | 0.338 | 8.84 | 0.296 | 5.79 |
| +/-80% (edge, balanced) | **0.223** | **2.25** | **0.163** | **1.75** |
| Blackout | 0.208 | 2.66 | 0.179 | 2.31 |

Averaged across both truths, `+/-80% (edge, balanced)` narrows `b_sd` by
45.7% against Blackout's 45.4% -- a dead heat on that measure, and it beats
Blackout on `valley_%` at both truths outright (2.25 vs 2.66, and 1.75 vs
2.31). This is the notebook-09 surprise, resolved: Blackout doesn't have a
special power to identify curvature that a continuous shape structurally
lacks. It only looked that way because the continuous option on the table
was the wasteful `uniform` draw.

## 4. Adstock robustness

Same decay-grid method as notebook 08, `white_noise` only (the corner
matching sections 2-3 and notebook 07), decay 0.0 to 0.7. The fitted model
is correctly specified for carryover throughout, same as notebook 08 -- this
isolates the lever's own robustness, not an analyst's mistake.

In [ ]:
def adstocked_frame(spend, decay):
    if decay == 0.0:
        return spend
    out = spend.copy()
    for ch in CHANNELS:
        out[ch] = apply_adstock(spend[ch].to_numpy(), decay)
    return out


ad_rows = []
for decay in DECAYS:
    for label, spec, nudge_shape, balance_signs in LEVERS:
        bias = {ch: [] for ch in CHANNELS}
        for demand_seed in range(AD_N_DEMAND_SEEDS):
            plan_df, dem, cal = build_world(demand_seed)
            seeds = [0] if is_unphased(spec) else range(AD_N_PHASING_SEEDS)
            means = {ch: [] for ch in CHANNELS}
            for phasing_seed in seeds:
                sched = schedule_for(plan_df, spec, phasing_seed, nudge_shape, balance_signs)
                dem_arr = dem.loc[sched.index].to_numpy()
                design_frame = adstocked_frame(sched, decay)
                draws = {ch: [] for ch in CHANNELS}
                for sim in range(AD_N_SIMS):
                    sales = simulate_sales(
                        sched, TRUE_MR, base_sales=cal.baseline_level, seed=sim,
                        demand=dem_arr, demand_coef=cal.demand_coef, adstock=decay,
                    )
                    fitted = fit_ols(design_frame, sales, controls=None)
                    for ch in CHANNELS:
                        draws[ch].append(fitted[ch])
                for ch in CHANNELS:
                    means[ch].append(np.mean(draws[ch]))
            for ch in CHANNELS:
                bias[ch].append(100 * (np.mean(means[ch]) - TRUE_MR[ch]) / TRUE_MR[ch])
        ad_rows.append({
            "decay": decay, "lever": label,
            "mean_bias_%": float(np.mean([np.mean(bias[ch]) for ch in CHANNELS])),
        })
        print(f"  decay={decay} {label}", flush=True)

adstock_frame_ = pd.DataFrame(ad_rows)
order = [lab for lab, _, _, _ in LEVERS]
piv = adstock_frame_.pivot(index="lever", columns="decay", values="mean_bias_%").loc[order]
removed = 100 * (piv.loc["unphased"] - piv) / piv.loc["unphased"]
print("\nbias removed vs unphased, %:")
print(removed.round(2).to_string())
print("\nshare of the no-carryover gain surviving at each decay, %:")
print((100 * removed.div(removed[0.0], axis=0)).round(2).to_string())

**Real numbers (`FAST_MODE=False`), decay=0.7:** `+/-80% (edge, balanced)`
removes 73.7% of bias and retains 90.4% of its own no-carryover gain --
against `+/-80% (uniform)`'s 43.2% / 74.3% and Blackout's 60.3% / 80.7%.
**Edge+balanced wins here too, on both the raw removal and the robustness-
to-decay measure.** `+/-40% (edge, balanced)` again lands close to
`+/-80% (uniform)` (39.3% removed vs 43.2%, but a better 79.0% survival
rate) at half the nominal setting.

## 5. Cost

Notebook 06 measured cost phasing TV alone; every other section in this
notebook phases all three channels together, so the cost comparison is
extended to match -- same `concave_truth` construction (contribution
`k * spend**b`, calibrated so the slope at each channel's own plan mean
equals its true marginal return), `b=0.6`, summed across channels. No
simulation needed here -- the concave truth is a deterministic function of
spend, so this is exact given the schedule draws.

In [ ]:
plan_df0 = build_world(demand_seed=0)[0]
X0 = {ch: float(plan_df0[ch].mean()) for ch in CHANNELS}
B_COST = 0.6


def concave_truth(x, ch, b):
    mr0 = TRUE_MR[ch]
    x0 = X0[ch]
    k = mr0 / (b * x0 ** (b - 1.0))
    return k * np.asarray(x, dtype=float) ** b


baseline_revenue = sum(concave_truth(plan_df0[ch].to_numpy(), ch, B_COST).sum() for ch in CHANNELS)

cost_rows = []
for label, spec, nudge_shape, balance_signs in LEVERS:
    if is_unphased(spec):
        totals = [baseline_revenue]
    else:
        totals = []
        for seed in range(N_DRAW_SEEDS):
            sched = schedule_for(plan_df0, spec, seed, nudge_shape, balance_signs)
            totals.append(sum(concave_truth(sched[ch].to_numpy(), ch, B_COST).sum() for ch in CHANNELS))
    mean_total = float(np.mean(totals))
    cost_rows.append({
        "lever": label,
        "cost: revenue given up %": 100 * (baseline_revenue - mean_total) / baseline_revenue,
    })

cost = pd.DataFrame(cost_rows).set_index("lever")
print(f"b={B_COST}, whole plan (all 3 channels phased together)\n")
print(cost.round(3).to_string())

**Real numbers (`FAST_MODE=False`):** `+/-80% (edge, balanced)` costs 8.89%
of revenue against Blackout's 9.92% -- about 10% cheaper, consistent with
notebook 06's TV-alone finding of 8.96% vs 9.98% at the same `b`.
`+/-40% (edge, balanced)` costs only 1.81%, cheaper than even
`+/-80% (uniform)`'s 2.29%.

## Bringing it together

All four axes, one table -- now including `+/-80% (annulus, balanced)`,
the middle shape, to settle whether it beats `edge` on cost for similar
rigor (notebook 06 found this for variance alone; see the synthesis
below for whether it holds up here). `+/-80% (edge, balanced)` isn't
just competitive with Blackout -- it **dominates** it: better on bias,
variance, saturation and adstock, and cheaper.

In [ ]:
summary = pd.DataFrame({
    "bias: removed %": {
        "+/-80% (uniform)": 58.21, "+/-40% (edge, balanced)": 48.65,
        "+/-80% (edge, balanced)": 81.18, "+/-80% (annulus, balanced)": 75.55,
        "Blackout": 74.11,
    },
    "variance: CV narrowed %": {
        "+/-80% (uniform)": 57.18, "+/-40% (edge, balanced)": 56.85,
        "+/-80% (edge, balanced)": 72.25, "+/-80% (annulus, balanced)": 66.44,
        "Blackout": 62.86,
    },
    "saturation: b_sd narrowed %, avg both truths": {
        "+/-80% (uniform)": 14.6, "+/-40% (edge, balanced)": 10.6,
        "+/-80% (edge, balanced)": 45.7, "+/-80% (annulus, balanced)": 26.5,
        "Blackout": 45.4,
    },
    "adstock: % of gain surviving at decay=0.7": {
        "+/-80% (uniform)": 74.34, "+/-40% (edge, balanced)": 79.04,
        "+/-80% (edge, balanced)": 90.43, "+/-80% (annulus, balanced)": 70.89,
        "Blackout": 80.74,
    },
    "cost: revenue given up %": {
        "+/-80% (uniform)": 2.285, "+/-40% (edge, balanced)": 1.808,
        "+/-80% (edge, balanced)": 8.885, "+/-80% (annulus, balanced)": 4.675,
        "Blackout": 9.915,
    },
})[["bias: removed %", "variance: CV narrowed %", "saturation: b_sd narrowed %, avg both truths",
    "adstock: % of gain surviving at decay=0.7", "cost: revenue given up %"]]
summary = summary.loc[["+/-80% (uniform)", "+/-40% (edge, balanced)", "+/-80% (edge, balanced)",
    "+/-80% (annulus, balanced)", "Blackout"]]
pd.set_option("display.width", 220)
print(summary.round(1).to_string())

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
colors = {
    "+/-80% (uniform)": "#9ca3af",
    "+/-40% (edge, balanced)": "#60a5fa",
    "+/-80% (edge, balanced)": "#059669",
    "+/-80% (annulus, balanced)": "#f59e0b",
    "Blackout": "#111827",
}
for ax, col in zip(axes, summary.columns):
    vals = summary[col]
    ax.bar(vals.index, vals, color=[colors[lv] for lv in vals.index])
    ax.set_title(col, fontsize=8)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
fig.suptitle("+/-80% (edge, balanced) beats Blackout on every rigor axis, and costs less")
plt.tight_layout()
plt.show()


**Blackout is not the answer. `edge`+`balance_signs=True` at the same
nominal cap is -- and it needs no dark weeks at all.** This overturns
notebook 10's benefit table, which was built entirely on `uniform` draws:
the "four measured rigor columns all pick Blackout, only cost cuts the
other way" story was comparing Blackout against a strawman. The real
comparison is between Blackout and the efficient continuous shape notebook
06 already knew about, and on every axis this notebook could re-measure,
the efficient shape wins.

**What this means for the three things downstream, none touched here:**

- **`notebooks/10_benefit_table.ipynb` needs a real rewrite, not a footnote.**
  Its whole premise -- one lever wins every measured axis, cost is the only
  tradeoff -- is still true, but the winning lever is `edge+balanced`, not
  Blackout. The benefit table should be rebuilt around the numbers in this
  notebook's summary table above.
- **`recommend_levers()` should very likely default to `edge`+`balance_signs`
  rather than choosing between `uniform` and `Blackout`.** Its current
  per-channel choice compares exactly the two options this notebook shows
  are both dominated by a third, better one.
- **`overview.html`'s headline scenario** — currently all-Blackout, per
  notebook 06's original flag — has an even stronger case for changing now:
  not just "off the cost frontier" but dominated outright, on the same
  bias/variance/identifiability claims the page is trying to make.

**Follow-up, now resolved: `annulus`+balanced does not beat `edge`+balanced.**
`+/-80% (annulus, balanced)` is cheaper than `edge+balanced` -- 4.68% vs
8.89%, roughly half -- but it gives up real rigor to get there, not just a
little. It narrows `b_sd` by 26.5% on average, against `edge`'s 45.7% and
even Blackout's 45.4%; and only 70.9% of the no-carryover adstock gain
survives at decay=0.7, against `edge`'s 90.4% and Blackout's 80.7% --
`annulus` is the *worst* of the four levers on adstock survival, uniform
included. It does hold its own on bias and variance (75.6% bias removed,
66.4% CV narrowed -- both close to Blackout, though still behind `edge`).
Notebook 06's finding that `annulus` was the more cost-efficient shape held
for the variance/cost frontier it measured; it does not extend to
saturation or adstock, the two axes where shape choice turns out to matter
most. `+/-80% (edge, balanced)` remains the recommendation -- it is not
just the best on rigor, it is also cheaper than Blackout, so there is no
cost-driven reason to trade down to `annulus` either.
